# FAKE NEWES DETECTION APP (for text like input)




In [38]:
!pip install streamlit pyngrok -q
!pip install python -u spacy -q
!python -m spacy download en_core_web_lg


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 800.0 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [45]:
# So from previous learning model we find out
# Logistic Regression through CountVectorizer as vectorizer, Spacy as text preprocessing function;
# Naive Bayes Classifier throuugh CountVectorizer as vectorizer, NLTK as text preprocessing function;
# Naive Bayes Classifier throuugh CountVectorizer as vectorizer, Spacy as text preprocessing function;
# acts as best
# and in case for choosing Prediction Method both Voting and MaxProbability both work as SAME
# so we use Voting...
# about the parameters they are precisely chosen manually.
%%writefile app.py
import streamlit as st
import re
import nltk
import spacy
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from transformers import BertTokenizer, BertModel
import torch
from tqdm import tqdm
from collections import Counter

# Download resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize NLP tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
nlp = spacy.load('en_core_web_lg')

# Preprocessing functions
def preprocess_text_nltk(text):
  text = re.sub(r'https?://\S+|www.\S+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', '', text)
  text = re.sub(r'\s+', ' ', text)
  text = text.lower()
  tokens = text.split()
  tokens = [lemmatizer.lemmatize(stemmer.stem(w)) for w in tokens if w not in stop_words]
  return ' '.join(tokens)

def preprocess_text_spacy(text):
  text = re.sub(r'https?://\S+|www.\S+', '', text)
  doc = nlp(text.lower())
  tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha and len(token.text) > 1]
  return ' '.join(tokens)

# --- Streamlit App ---
st.title("Fake News Detection App")

st.title(" Upload Training Files")

# Upload train.csv
train_file = st.file_uploader("Upload train(1).csv", type="csv")
# Upload test.csv
test_file = st.file_uploader("Upload test(1).csv", type="csv")
# Upload sample_submission.csv
sub_file = st.file_uploader("Upload sample_submission.csv", type="csv")
# Enter text
st.write("Enter a tweet or headline to check if it's Real or Fake.")
user_input = st.text_area("Input text here:")

if train_file and test_file and sub_file:
    # Read files
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    sub_df = pd.read_csv(sub_file)

    # Combine
    X_train_text = train_df['text'].fillna("").tolist() + test_df['text'].fillna("").tolist()
    y_train = train_df['target'].tolist() + sub_df['target'].tolist()

    st.success("✅ Files loaded and combined!")
else:
  st.warning("⚠️Please upload all files.")
  st.stop
if st.button("Predict"):
    if user_input:
        # Preprocess training data
        X_train_nltk = [preprocess_text_nltk(t) for t in X_train_text]
        X_train_spacy = [preprocess_text_spacy(t) for t in X_train_text]

        # Vectorizers
        cv_nltk = CountVectorizer(max_features = 5000, min_df = 2, max_df = 0.97, stop_words = 'english')
        cv_spacy = CountVectorizer(max_features = 5000, min_df = 2, max_df = 0.97, stop_words = 'english')
        X_vec_cv_nltk = cv_nltk.fit_transform(X_train_nltk)
        X_vec_cv_spacy = cv_spacy.fit_transform(X_train_spacy)

        # Train models
        model_lr_cv_spacy = LogisticRegression(C=1, max_iter=1000, solver='saga')
        model_lr_cv_spacy.fit(X_vec_cv_spacy, y_train)
        model_nb_cv_nltk = MultinomialNB(alpha=1)
        model_nb_cv_nltk.fit(X_vec_cv_nltk, y_train)
        model_nb_cv_spacy = MultinomialNB(alpha=1)
        model_nb_cv_spacy.fit(X_vec_cv_spacy, y_train)

        # Preprocess user input
        text_nltk = preprocess_text_nltk(user_input)
        text_spacy = preprocess_text_spacy(user_input)

        # Vectorize
        vec_cv_nltk = cv_nltk.transform([text_nltk])
        vec_cv_spacy = cv_spacy.transform([text_spacy])

        # Predictions
        pred_lr_cv_spacy = model_lr_cv_spacy.predict(vec_cv_spacy)[0]
        pred_nb_cv_nltk = model_nb_cv_nltk.predict(vec_cv_nltk)[0]
        pred_nb_cv_spacy = model_nb_cv_spacy.predict(vec_cv_spacy)[0]

        # Majority voting
        votes = [pred_lr_cv_spacy, pred_nb_cv_nltk, pred_nb_cv_spacy]
        final_pred = Counter(votes).most_common(1)[0][0]
        confidence = (votes.count(final_pred) / 3) * 100

        result = "Real News" if final_pred == 1 else "Fake News"
        st.subheader(f"Prediction: {result}")
        st.write(f"Confidence: {confidence:.2f}%")
    else:
        st.warning("Please enter some text.")

Overwriting app.py


In [46]:
!kill $(ps aux | grep streamlit | awk '{print $2}')
!kill $(ps aux | grep ngrok | awk '{print $2}')

^C
^C


In [47]:
from pyngrok import conf, ngrok
import os
# Step 1: Set your authtoken
conf.get_default().auth_token = "2z7pwLR81mQEv5MF71rDRbU64iw_6s1LiR3wAhm5NoEAar86K"

# Step 2: Start Streamlit app in back ground
!streamlit run app.py &> /dev/null &

# Step 3: Create tunnel
public_url = ngrok.connect("http://localhost:8501")
print("Your app is live at:", {public_url})


Your app is live at: {<NgrokTunnel: "https://01a0-34-67-45-250.ngrok-free.app" -> "http://localhost:8501">}
